# 03 · Automated variance flagging

**Goal:** each month, tell budget owners which of the 27 lines need attention, and draft the explanation.

| Status | Rule |
|---|---|
| 🔴 **Red: investigate** | **material** vs budget (≥ $ **and** ≥ % threshold) **and** outside the forecast's **expected range** |
| 🟠 **Amber: explain** | material but expected, a small surprise, or off budget in one direction for 3+ months |
| 🟢 **Green: on track** | none of the above |

Thresholds are set per line in `config/variance_rules.yaml`, for example $150k and 3% for subscription revenue.

**Key findings**
1. Materiality plus statistical surprise cuts 27 lines to a short "investigate" list.
2. Each flag comes with draft commentary and a drill-down to journal entries.
3. Scored against a hidden answer key, Red flags catch the large majority of planted problems.

In [1]:
import json
from pathlib import Path

import pandas as pd

from fpa.viz import usd

pd.set_option("display.max_colwidth", 200)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MARTS = ROOT / "data" / "marts"
read = lambda name: pd.read_parquet(MARTS / f"{name}.parquet")
meta = json.loads((MARTS / "run_metadata.json").read_text())
AS_OF = meta["as_of"]
flags = read("variance_monthly")

## 1. What needs attention this month?

In [2]:
cur = flags[flags["period"] == AS_OF]
print(f"{AS_OF}: " + ", ".join(f"{s} {int((cur['severity'] == s).sum())}" for s in ("Red", "Amber", "Green")))
cols = ["severity", "fs_line", "department", "actual_usd", "budget_usd", "var_budget_usd", "var_budget_pct"]
(cur[cur["severity"] != "Green"].sort_values("impact_usd")[cols]
 .style.format({"actual_usd": usd, "budget_usd": usd, "var_budget_usd": usd, "var_budget_pct": "{:+.1%}"}).hide(axis="index"))

2026-08: Red 0, Amber 7, Green 20


severity,fs_line,department,actual_usd,budget_usd,var_budget_usd,var_budget_pct
Amber,Depreciation,G&A,$154k,$127k,$26k,+20.8%
Amber,Services Revenue,Company,$527k,$504k,$23k,+4.5%
Amber,Personnel,Sales,$781k,$847k,-$66k,-7.8%
Amber,Personnel,G&A,$479k,$549k,-$70k,-12.7%
Amber,Commissions,Sales,$118k,$229k,-$111k,-48.6%
Amber,Personnel,Engineering,$1.5M,$1.6M,-$141k,-8.6%
Amber,Subscription Revenue,Company,$5.9M,$5.7M,$172k,+3.0%


Red lines need investigating. Amber lines deserve a sentence in the pack, not a deep dive.

## 2. Why is a line flagged?

Each flag lists its top drivers compared with a typical recent month. Here is the latest Red flag, from commentary to journal entries.

In [3]:
red = flags[flags["severity"] == "Red"].sort_values("period")
example = red[red["fs_line"] == "Subscription Revenue"].iloc[-1] if (red["fs_line"] == "Subscription Revenue").any() else red.iloc[-1]
print(example["commentary"], "\n")
drivers = pd.DataFrame(json.loads(example["drivers"]))
drivers.style.format({"current_usd": usd, "typical_usd": usd, "change_usd": usd}).hide(axis="index")

Subscription Revenue was $410k (7%) under budget in Jun-26 (unfavourable). The actual of $5.08M fell outside the expected range of $5.58M to $5.96M from last month's forecast - likely a one-off or error to investigate. Biggest change vs a typical recent month: Credit memo - billing dispute - Northridge Logistics Inc. -$607k (1 document). 



driver,current_usd,typical_usd,change_usd,documents
Credit memo - billing dispute - Northridge Logistics Inc.,-$607k,$0,-$607k,1
Recurring subscription billing,$5.7M,$5.5M,$226k,3


In [4]:
detail = read("gl_detail_recent")
d = detail[(detail["fs_line"] == example["fs_line"]) & (detail["department"] == example["department"]) & (detail["period"] == example["period"])]
d.reindex(d["amount_usd"].abs().sort_values(ascending=False).index).head(5)[["posting_date", "description", "source", "amount_usd"]].style.format({"amount_usd": usd}).hide(axis="index")

posting_date,description,source,amount_usd
2026-06-22 00:00:00,Credit memo - billing dispute - Northridge Logistics Inc.,AR,-$607k
2026-06-30 00:00:00,Subscription revenue - Stonebridge Pharma,REVREC,$61k
2026-06-30 00:00:00,Subscription revenue - Summitlight Pharma,REVREC,$49k
2026-06-30 00:00:00,Subscription revenue - Prairiestack Insurance,REVREC,$47k
2026-06-30 00:00:00,Subscription revenue - Ironshore Energy,REVREC,$42k


Payroll flags also split the variance into headcount and pay rate:

In [5]:
pers = flags[(flags["fs_line"] == "Personnel") & (flags["personnel_split"] != "")].sort_values("period").iloc[-1]
print(pers["commentary"])

Personnel (Sales) was $66k (8%) under budget in Aug-26 (favourable). It was in line with last month's forecast, so the gap reflects the original plan rather than a surprise. This is the fifth consecutive month under budget. Headcount was 62 vs a plan of 67 (-5 heads = -$63k); cost per head accounted for -$3k. Biggest change vs a typical recent month: Payroll run +$10k (4 documents).


## 3. Does it work?

The data generator planted problems (duplicate invoices, missed accruals, cloud-spend spikes, credit memos) in a hidden answer key that the pipeline never reads, so I can score the flags:
- **Recall:** the share of planted problems flagged Red.
- **Precision:** the share of Red flags that were real problems.

In [6]:
ev = read("flag_evaluation")
summary = meta["variance"]["evaluation"]
print(f"Window {summary['window']}: {summary['anomalies_in_window']} injected anomalies, {summary['red_flags']} Red flags")
print(f"  Recall (Red):            {summary['recall_red']:.0%}")
print(f"  Precision (Red):         {summary['precision_red']:.0%}")
print(f"  Recall (Red or Amber):   {summary['recall_red_or_amber']:.0%}")
ev[["period", "type", "series", "approx_usd", "best_flag"]].style.format({"approx_usd": lambda v: "" if pd.isna(v) else usd(v)}).hide(axis="index")

Window 2025-09..2026-08: 13 injected anomalies, 17 Red flags
  Recall (Red):            92%
  Precision (Red):         82%
  Recall (Red or Amber):   100%


period,type,series,approx_usd,best_flag
2025-09,legal_settlement,Professional Fees | G&A,$247k,Amber
2025-10,cloud_cost_spike,Hosting | Cloud Operations,$253k,Red
2025-10,miscoded_cost_center,"Contractors | Engineering, Contractors | Professional Services",$92k,Red
2026-02,unbudgeted_contractors,Contractors | G&A,$148k,Red
2026-02,unbudgeted_contractors,Contractors | G&A,$115k,Red
2026-03,software_auto_renewal,Software & Tools | G&A,$187k,Red
2026-03,cloud_cost_spike,Hosting | Cloud Operations,$203k,Red
2026-04,legal_settlement,Professional Fees | G&A,$341k,Red
2026-04,revenue_credit_memo,Subscription Revenue | Company,$461k,Red
2026-05,cloud_cost_spike,Hosting | Cloud Operations,$252k,Red


- **Misses** tend to be one-offs on noisy lines, such as legal fees, that land inside the wide expected range and only reach Amber.
- **False positives** are genuine but unusual swings, still worth a quick look.
- Lower thresholds catch more at the cost of more noise. FP&A owns that dial in the YAML file.

Results refresh after every close in the [live dashboard](https://fpa-ml-forecast.streamlit.app).